```mermaid
flowchart TD
    A([START]) --> B[User gửi câu hỏi]
    B --> C[LLM phân tích ý định]
    C --> D{Có cần gọi tool?}

    D -- Không --> E[Trả lời trực tiếp]
    D -- Có --> F["Gọi tool: get_weather(city)"]
    F --> G[Tool trả kết quả]
    G --> H[LLM tổng hợp câu trả lời]
    H --> I([END])

    E --> I
```

In [1]:
import operator
from typing_extensions import TypedDict, Annotated

In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import AnyMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END

In [ ]:
# ===== Tool =====
@tool
def get_weather(city: str) -> str:
    """Lấy thời tiết cho một thành phố."""
    data = {
        "hanoi": "Hà Nội: 24°C, nhiều mây",
        "danang": "Đà Nẵng: 29°C, nắng đẹp",
        "hcm": "TP.HCM: 32°C, có mưa nhẹ",
    }
    return data.get(city.lower(), f"Chưa có dữ liệu thời tiết cho {city}")

tools = [get_weather]
tools_by_name = {t.name: t for t in tools}

In [ ]:
# ===== Model =====
model = init_chat_model("claude-sonnet-4-6", temperature=0)
model_with_tools = model.bind_tools(tools)

In [ ]:
# ===== State =====
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [ ]:
# ===== Nodes =====
def llm_node(state: AgentState):
    system = SystemMessage(
        content=(
            "Bạn là trợ lý hữu ích. "
            "Nếu người dùng hỏi thời tiết và có nêu thành phố, hãy gọi tool get_weather. "
            "Nếu chưa rõ thành phố, hãy hỏi lại ngắn gọn."
        )
    )
    response = model_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [ ]:
# ===== Nodes =====
def tool_node(state: AgentState):
    last_msg = state["messages"][-1]
    tool_messages = []

    for call in last_msg.tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        tool = tools_by_name[tool_name]
        tool_result = tool.invoke(tool_args)

        tool_messages.append(ToolMessage(
            content=str(tool_result),
            tool_call_id=call["id"],
        ))

    return {"messages": tool_messages}

In [ ]:
# ===== Conditional edge =====
def should_continue(state: AgentState):
    last_msg = state["messages"][-1]
    if getattr(last_msg, "tool_calls", None):
        return "tool_node"
    return END

In [ ]:
# ===== Build graph =====
builder = StateGraph(AgentState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", should_continue)
builder.add_edge("tool_node", "llm_node")

graph = builder.compile()

In [ ]:
# ===== Run =====
result = graph.invoke({"messages": [
    {"role": "user", "content": "Thời tiết ở Hanoi hôm nay thế nào?"}
]})
result

In [ ]:
for msg in result["messages"]:
    print(msg)

In [ ]:
# ===== Run =====
result = graph.invoke({"messages": [
    {"role": "user", "content": "Nhiệt độ hôm nay thế nào?"}
]})
result

In [ ]:
for msg in result["messages"]:
    print(msg)